# 💓 Notebook 1: Fixed-Interval Heartbeats

**The big question:** *"How does one machine know that another machine is still alive?"*

The basic idea is simple: every node sends a small "I'm alive!" message — a **heartbeat** — every few seconds. If a peer doesn't hear a heartbeat for some timeout window, it declares the sender dead and reroutes work elsewhere.

Real systems that use this pattern:

| System | Who sends | Who listens | Default cadence |
|---|---|---|---|
| **HDFS** | DataNode | NameNode | every 3s, dead after ~10 missed |
| **Kubernetes** | kubelet on each node | kube-controller-manager | every 10s, `NotReady` after 40s |
| **Cassandra** | every node | every other node (gossip) | every 1s |
| **GFS** | ChunkServer | Master | piggy-backed with instructions |

All of these have to choose: **how long do we wait before we panic?**

- Short timeout → fast failure detection, **lots of false positives** (a slow network looks like a dead node).
- Long timeout → reliable, **slow** to react to real failures.

In this notebook we simulate a heartbeat sender + monitor and feel the trade-off. In notebook 2 we'll fix the trade-off with the **phi accrual** detector. In notebook 3 we'll look at *how* heartbeats are usually wired up in real systems (push vs pull, central vs gossip).

## Learning objectives
- Implement a send/receive heartbeat loop with timestamps.
- See the false-positive vs detection-delay trade-off in numbers and a plot.
- Understand the common **"k missed beats"** rule and why a single missed beat is rarely enough.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/heartbeat
uv sync
```

Then in VS Code, click the kernel picker (top-right of the notebook) and select the `.venv` kernel. If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 1. Generate a heartbeat trace

We pretend a node sends a heartbeat every `TICK = 1.0` seconds, but the network adds some random jitter (latency wobble). At `t = 30s` the node truly dies and stops sending.

We collect a list of timestamps `heartbeats` — these are the moments a heartbeat *arrived at the monitor*.

In [ ]:
import random
random.seed(0)

TICK    = 1.0   # heartbeat every ~1s
JITTER  = 0.4   # +/- 0.4s of random network jitter
DEAD_AT = 30.0  # node dies at this time (no more heartbeats after this)
TOTAL   = 60.0  # how long the monitor watches

heartbeats = []
t = 0.0
while t < TOTAL:
    t += TICK + random.uniform(-JITTER, JITTER)
    if t >= DEAD_AT:
        break  # node is dead, no more beats
    heartbeats.append(t)

print(f'node sent {len(heartbeats)} heartbeats')
print(f'last heartbeat received at t={heartbeats[-1]:.2f}s')
print(f'node actually died at t={DEAD_AT}')
gaps = [round(b-a,2) for a,b in zip(heartbeats, heartbeats[1:])][:5]
print(f'first 5 inter-arrival gaps: {gaps}')

## 2. A naive monitor with a fixed timeout

The monitor walks forward in time. At every sample it asks: *"how long since I last heard a heartbeat?"* If that gap exceeds `timeout`, it declares the node dead.

We track two outcomes:
- **detection_delay** — how many seconds after the *real* death we noticed.
- **false_positives** — how many times we briefly declared the node dead **while it was actually alive** (these are the pages that wake your on-call at 3am for nothing).

In [ ]:
def simulate_fixed_timeout(heartbeats, timeout, dead_at=DEAD_AT,
                          total=TOTAL, sample_every=0.1):
    """Walk time forward; declare 'dead' when gap > timeout."""
    last_seen = None         # None until the first beat arrives
    declared_dead_at = None  # first time we declared the node dead AFTER it really died
    false_positives = 0      # times we flipped to 'dead' while node was alive
    currently_dead = False
    hb_iter = iter(heartbeats)
    next_hb = next(hb_iter, None)

    t = 0.0
    while t < total:
        # Deliver any heartbeats that have arrived by time t
        while next_hb is not None and next_hb <= t:
            last_seen = next_hb
            currently_dead = False  # heartbeat = node is alive again
            next_hb = next(hb_iter, None)

        # Only judge liveness once we've heard at least one heartbeat
        if last_seen is not None and (t - last_seen) > timeout:
            if not currently_dead:
                currently_dead = True
                if t < dead_at:
                    false_positives += 1     # flapped while still alive
            # Detection = the detector says DOWN at some point at/after the real death
            # and never recovers. Note this is deliberately NOT `elif`: with a short
            # timeout the flip can happen just BEFORE dead_at (the node had already
            # gone quiet), and that same flip is the detection. Treating those cases
            # as "never detected" is what leaves gaps in the curve below.
            if t >= dead_at and declared_dead_at is None:
                declared_dead_at = t
        t += sample_every

    # If the detector was already DOWN when the node died, credit the detection to
    # the moment of death: it needed 0 extra seconds to notice.
    detection_delay = (declared_dead_at - dead_at) if declared_dead_at is not None else None
    return declared_dead_at, detection_delay, false_positives

header = f"{'timeout':>8} | {'declared_dead_at':>16} | {'detection_delay':>15} | {'false_positives':>15}"
print(header)
print('-' * len(header))
rows = {}
for to in (0.5, 1.0, 1.5, 2.0, 3.0, 5.0):
    dead_at_s, delay, fp = simulate_fixed_timeout(heartbeats, to)
    rows[to] = (delay, fp)
    dead_str  = f'{dead_at_s:.2f}'  if dead_at_s is not None else 'None'
    delay_str = f'{delay:.2f}'      if delay   is not None else 'None'
    print(f'{to:>8.1f} | {dead_str:>16} | {delay_str:>15} | {fp:>15}')

# Every timeout must eventually notice a node that stopped sending forever.
assert all(d is not None for d, _ in rows.values()), rows
# The trade-off, as an invariant: false positives fall as the timeout grows,
# detection delay rises. Neither is optional; that is the whole problem.
tos = sorted(rows)
fps = [rows[t][1] for t in tos]
delays = [rows[t][0] for t in tos]
assert fps == sorted(fps, reverse=True), fps
assert delays == sorted(delays), delays
# And there is no free lunch here: the only timeouts with zero false positives are
# the slow ones.
clean = [t for t in tos if rows[t][1] == 0]
assert clean and min(clean) >= 1.5, clean
assert rows[0.5][1] > 5, 'a half-tick timeout should flap constantly'
print(f'\n✔ false positives: {fps[0]} at timeout=0.5s down to {fps[-1]} at 5.0s')
print(f'  detection delay:  {delays[0]:.2f}s at 0.5s up to {delays[-1]:.2f}s at 5.0s')
print(f'  cheapest timeout with zero false alarms: {min(clean)}s '
      f'(costs {rows[min(clean)][0]:.2f}s of detection delay)')

### Reading the table

- **timeout = 0.5s** — half a tick. With 0.4s jitter we miss the deadline constantly: dozens of false positives.
- **timeout = 1.0s** — exactly one tick. Still flaps because jitter pushes some gaps over 1.0s.
- **timeout = 1.5s** — beats the jitter envelope; usually clean during life but ~1.5s slower to detect a real death.
- **timeout = 5.0s** — rock solid, but we wait an extra ~5s in an outage to do anything about it.

In production you don't get to know the jitter ahead of time, and it changes (deploy traffic, GC pause, noisy neighbour…). That's why a fixed timeout is fragile.

## 3. Visualize the trade-off

Let's run many timeouts and plot **false positives** and **detection delay** on the same x-axis. You should see them move in opposite directions — the classic Pareto curve of failure detection.

In [ ]:
import matplotlib.pyplot as plt

timeouts = [round(0.4 + 0.1*i, 2) for i in range(0, 47)]  # 0.4s .. 5.0s
fps, delays = [], []
for to in timeouts:
    _, delay, fp = simulate_fixed_timeout(heartbeats, to)
    fps.append(fp)
    delays.append(delay if delay is not None else float('nan'))

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(timeouts, fps, 'o-', color='tab:red', label='false positives (during life)')
ax1.set_xlabel('timeout (seconds)')
ax1.set_ylabel('false positives', color='tab:red')
ax1.tick_params(axis='y', labelcolor='tab:red')

ax2 = ax1.twinx()
ax2.plot(timeouts, delays, 's-', color='tab:blue', label='detection delay')
ax2.set_ylabel('detection delay (s)', color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')

plt.title('Fixed timeout: pick your poison')
fig.tight_layout()
plt.show()

# The two curves must move in opposite directions across the whole sweep — that is
# what makes this a trade-off rather than a tuning problem with a right answer.
assert all(b <= a for a, b in zip(fps, fps[1:])), 'false positives should fall as timeout grows'
assert all(b >= a for a, b in zip(delays, delays[1:])), 'delay should rise as timeout grows'

# The decisive check: no single timeout achieves both the best false-positive count
# AND the best detection delay. If one did, there would be nothing to trade.
best_fp, best_delay = min(fps), min(delays)
assert not any(fp == best_fp and d == best_delay for fp, d in zip(fps, delays)), \
    'some timeout is optimal on both axes — then this would not be a trade-off'

# Quantify the cheapest "no false alarms" setting.
zero_fp = [(to, d) for to, fp, d in zip(timeouts, fps, delays) if fp == 0]
cheapest_to, cost = min(zero_fp)
print(f'✔ across {len(timeouts)} timeouts from {timeouts[0]}s to {timeouts[-1]}s:')
print(f'  false positives fall {fps[0]} -> {fps[-1]}, detection delay rises '
      f'{delays[0]:.2f}s -> {delays[-1]:.2f}s')
print(f'  the cheapest timeout with zero false alarms is {cheapest_to}s, '
      f'and it costs {cost:.2f}s of detection delay')

## 4. The "k missed beats" rule

Real systems rarely use a single timeout. They use a rule like *"declare dead after **k** consecutive missed heartbeats"*. This is mathematically the same as `timeout = k * interval`, but it's friendlier to reason about:

- HDFS: missed heartbeats × interval ≈ **10 × 3s = 30s** before a DataNode is dead.
- Kubernetes node controller: `--node-monitor-grace-period=40s` with 10s heartbeats ≈ **4 misses**.
- TCP keepalive: defaults to **9 probes × 75s** on Linux — way too slow for service-level liveness, which is why apps add their own heartbeats.

The simulation below shows the same trade-off expressed in misses instead of seconds.

In [ ]:
k_rows = {}
for k in (1, 2, 3, 5, 10):
    timeout = k * TICK
    dead_at_s, delay, fp = simulate_fixed_timeout(heartbeats, timeout)
    k_rows[k] = (delay, fp)
    delay_str = f'{delay:.2f}s' if delay is not None else 'None'
    print(f'k={k:>2} missed beats (timeout={timeout}s) -> '
          f'detection_delay={delay_str}, false_positives={fp}')

# "k missed beats" is literally timeout = k * interval, so it inherits the same
# trade-off — renaming the knob does not remove it.
assert k_rows[1][1] > k_rows[3][1] >= k_rows[10][1], k_rows
assert k_rows[1][0] < k_rows[10][0], k_rows
assert k_rows[10][1] == 0
print(f'\n✔ k=1 flaps {k_rows[1][1]} times; k=10 never flaps but takes '
      f'{k_rows[10][0]:.1f}s to notice a real death')
print('  Same Pareto curve, friendlier units. Notebook 2 changes the shape of the curve.')

## 🤔 What we just saw

- Heartbeating is dead simple to implement: a periodic message + a timer on the receiver.
- The hard part is **picking the timeout**. There is no single right number — it depends on the network, on the GC pauses of your runtime, and on how expensive a false positive is (think: declaring a healthy primary dead and triggering a failover).
- One easy improvement is the **"k missed beats"** rule. It's still a fixed threshold, but it's easier to communicate.

👉 Next: notebook 2 replaces the hard threshold with an **adaptive suspicion score** that learns the network's normal cadence.